# Week 14 — Build a Local LLM Chatbot & Tool-Using Agent

**Theme:** LLMs & agentic AI

Every previous project trained a model from scratch on a small dataset. This
week is different: we use an already-trained **open-source language model** —
no training, no paid API, just Colab. Instead of training, we *prompt* it —
and then give it **tools** (Python functions it can decide to call) to build
a minimal **agent**.

**What "agentic AI" means, in one sentence:** instead of the model only
replying with text, we let it request that *we* run a function, look at the
result, and use it to keep working toward the answer — a loop of "think, act,
observe" instead of a single response.

**A note on model choice:** the same three ideas below (system prompts,
multi-turn memory, a tool-use agent loop) apply to *any* LLM you might call
later — a small local model here, or a large hosted model like Claude via a
paid API. We picked a free, local, open-source model so this notebook runs
with zero signup and zero cost. If you later plug in a paid API instead, only
the `generate()` function in the next cell would need to change — everything
built on top of it stays the same.

In [ ]:
!pip install -q transformers accelerate

## 0. Load a local open-source model (free, no signup)

We'll use **[Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)**,
a small (1.5-billion-parameter) instruction-tuned model that's free to
download and runs directly on Colab's own hardware — no API key, no account,
no cost. Being roughly 1000x smaller than a frontier model, it's noticeably
less capable (shorter memory, weaker reasoning, less reliable at following
complex instructions) — you'll see that trade-off in section 4.

**Tip:** this notebook works on CPU, but it's much faster with a GPU — in
Colab, go to *Runtime → Change runtime type → T4 GPU* before running the cell
below.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

def generate(messages, system=None, max_new_tokens=300):
    """Run one turn of chat through the local model and return the reply text."""
    chat = ([{"role": "system", "content": system}] if system else []) + messages
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 1. Your first generation

Every call goes through the same `generate()` helper: a list of
`{"role": ..., "content": ...}` messages in, a plain string reply out. (This
is deliberately the same shape hosted chat APIs use — `apply_chat_template`
is what turns it into the exact prompt format this specific model was
trained on.)

In [ ]:
reply = generate([{"role": "user", "content": "In two sentences, what is machine learning?"}])
print(reply)

## 2. System prompts: giving the model a persona

A `system` prompt sets standing instructions that apply to the whole
conversation — tone, role, constraints — separately from the user's actual
question.

In [ ]:
def ask(question, system=None):
    return generate([{"role": "user", "content": question}], system=system)

plain = ask("Explain what a neural network is.")
print("--- Default ---")
print(plain)

pirate = ask(
    "Explain what a neural network is.",
    system="You are a very enthusiastic pirate captain who explains everything using nautical metaphors.",
)
print("\n--- Pirate persona ---")
print(pirate)

## 3. A multi-turn conversation

The model itself has no memory between calls — *you* resend the whole
conversation history each time. We wrap that in a small helper class.

In [ ]:
class Chat:
    def __init__(self, system=None):
        self.system = system
        self.history = []

    def send(self, user_message):
        self.history.append({"role": "user", "content": user_message})
        reply = generate(self.history, system=self.system)
        self.history.append({"role": "assistant", "content": reply})
        return reply

chat = Chat(system="You are a friendly, concise tutor for first-year AI students.")
print(chat.send("My name is Minjun and I'm learning about RNNs."))
print()
print(chat.send("What's my name, and what was I just learning about?"))

## 4. Give the model tools: a minimal agent

The model can't run code by itself — but it *can* tell you "please run this
function with these arguments, then tell me the result." Hosted APIs like
Claude have a built-in `tools` parameter for this; our local model doesn't,
so we'll build the same idea ourselves with a simple convention: **when the
model wants to call a tool, it replies with one line of JSON instead of
prose.** This is actually close to what's happening "under the hood" of
every tool-calling API — worth seeing once.

We define two tools, describe them, and write the loop that executes
whichever tool the model asks for.

**Tool 1 — calculator.** We deliberately do NOT use Python's `eval()` on the
model's input (that would let it run arbitrary code) — instead we parse the
expression into a syntax tree and only allow basic arithmetic.

In [ ]:
import ast
import operator

_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.Pow: operator.pow, ast.USub: operator.neg,
}

def safe_calculate(expression):
    """Evaluate a simple arithmetic expression (+ - * / ** parentheses) safely."""
    def _eval(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsupported expression: {expression!r}")
    tree = ast.parse(expression, mode="eval")
    return _eval(tree.body)

# quick sanity check
print(safe_calculate("(12 + 8) * 3 - 5 ** 2"))

In [ ]:
def convert_units(value, from_unit, to_unit):
    """Convert between a small set of common units."""
    conversions = {
        ("km", "mi"): lambda v: v * 0.621371,
        ("mi", "km"): lambda v: v / 0.621371,
        ("kg", "lb"): lambda v: v * 2.20462,
        ("lb", "kg"): lambda v: v / 2.20462,
        ("celsius", "fahrenheit"): lambda v: v * 9 / 5 + 32,
        ("fahrenheit", "celsius"): lambda v: (v - 32) * 5 / 9,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key not in conversions:
        raise ValueError(f"Unsupported conversion: {from_unit} -> {to_unit}")
    return conversions[key](value)

print(convert_units(100, "km", "mi"))

Now we describe both functions as **tools** the model can choose to call —
`input_schema` documents exactly what arguments each tool expects (and
doubles as the text we show the model, since it has no native tools API).

In [ ]:
tools = [
    {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression using + - * / ** and parentheses.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "e.g. '(12 + 8) * 3'"},
            },
            "required": ["expression"],
        },
    },
    {
        "name": "convert_units",
        "description": "Convert a numeric value between units: km<->mi, kg<->lb, celsius<->fahrenheit.",
        "input_schema": {
            "type": "object",
            "properties": {
                "value": {"type": "number"},
                "from_unit": {"type": "string", "enum": ["km", "mi", "kg", "lb", "celsius", "fahrenheit"]},
                "to_unit": {"type": "string", "enum": ["km", "mi", "kg", "lb", "celsius", "fahrenheit"]},
            },
            "required": ["value", "from_unit", "to_unit"],
        },
    },
]

TOOL_FUNCTIONS = {
    "calculator": lambda input: safe_calculate(input["expression"]),
    "convert_units": lambda input: convert_units(input["value"], input["from_unit"], input["to_unit"]),
}

## 5. The agent loop

This is the core pattern behind every tool-using AI agent, hosted API or
local model alike:

1. Send the conversation (with the available tools described in the system
   prompt) to the model
2. If the reply is a tool-call JSON line, run that tool and send the result
   back as the next turn
3. Repeat until the model replies with plain text instead of a tool call (or
   we hit a step limit, in case it gets stuck)

In [ ]:
import json
import re

def _tool_signature(tool):
    props = tool["input_schema"].get("properties", {})
    args = ", ".join(f'{name}: {spec.get("type", "any")}' for name, spec in props.items())
    return f'{tool["name"]}({args})'

def _build_tools_system_prompt(tools):
    lines = [
        "You can use tools to help answer questions. To call a tool, reply with ONLY "
        "one line of JSON, exactly in this format (no other text before or after it):",
        '{"tool": "<tool_name>", "input": {<arguments>}}',
        "",
        "Available tools:",
    ]
    lines += [f"- {_tool_signature(t)}: {t['description']}" for t in tools]
    lines.append("")
    lines.append("If you already know the final answer, just reply in plain text (no JSON).")
    return "\n".join(lines)

def _extract_tool_call(text):
    """Try to parse a {"tool": ..., "input": ...} JSON call out of the model's reply."""
    candidates = [text.strip()]
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        candidates.append(match.group(0))
    for candidate in candidates:
        try:
            obj = json.loads(candidate)
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict) and "tool" in obj:
            return obj
    return None

def run_agent(user_message, verbose=True, max_steps=6):
    system = _build_tools_system_prompt(tools)
    messages = [{"role": "user", "content": user_message}]

    for _ in range(max_steps):
        reply = generate(messages, system=system)
        call = _extract_tool_call(reply)
        if call is None:
            return reply

        tool_name, tool_input = call.get("tool"), call.get("input", {})
        if verbose:
            print(f"[agent] calling tool '{tool_name}' with input {tool_input}")
        messages.append({"role": "assistant", "content": reply})

        if tool_name in TOOL_FUNCTIONS:
            try:
                result = TOOL_FUNCTIONS[tool_name](tool_input)
                result_text = f"Tool result: {result}"
            except Exception as e:
                result_text = f"Tool error: {e}"
        else:
            result_text = f"Tool error: unknown tool '{tool_name}'"
        messages.append({"role": "user", "content": result_text})

    return f"(gave up after {max_steps} steps) {reply}"

In [ ]:
answer = run_agent(
    "A road trip is 342 kilometers. If gas costs $1.85 per liter and the car "
    "uses 1 liter per 15 km, roughly how much will gas cost in total? Also, "
    "how far is that trip in miles?"
)
print("\nFinal answer:\n", answer)

Watch the printed `[agent] calling tool ...]` lines above: the model decided
*on its own*, from the plain-English question, which tools to call, in what
order, and how to combine the results — that decision-making loop is what
makes this an "agent" rather than a single API call.

**Heads-up:** a 1.5B-parameter model is much less reliable at this than a
frontier model — it may occasionally answer directly without calling a tool,
call the wrong tool, or produce badly-formed JSON our simple parser can't
catch (in which case `run_agent` just treats its reply as the final answer).
If that happens, re-run the cell — you're seeing a real capability trade-off,
not a bug.

## Try it yourself

1. **Add a third tool.** Write a `word_count(text)` function or a
   `roman_numeral(number)` converter, describe it in the `tools` list, add it
   to `TOOL_FUNCTIONS`, and ask a question that needs it.
2. **Break it on purpose.** Ask for a conversion the tool doesn't support
   (e.g. `"convert 10 gallons to liters"`) — does the agent handle the tool's
   error gracefully, or does it get stuck?
3. **Compare personas.** Write two very different system prompts (e.g. "a
   strict professor" vs. "a supportive coach") and ask the same question to
   both — how much does phrasing/tone change vs. the actual content?
4. **Swap in a bigger model.** Try changing `MODEL_NAME` to a larger instruct
   model (e.g. `"Qwen/Qwen2.5-7B-Instruct"`, if your Colab session has enough
   GPU memory) and re-run the agent demo — does tool selection get noticeably
   more reliable? This is the same trade-off you'd face choosing between a
   cheap and an expensive model on a paid API: more capable almost always
   costs more (here, in GPU memory and speed instead of dollars).

---
## 🎯 캡스톤: 나만의 학교생활 비서 에이전트

위에서는 계산기와 단위 변환 두 가지 도구만 가진 에이전트를 만들었습니다. 이번엔 **여러분만의 새로운 도구를 최소 1개 직접 설계**해서 에이전트에 추가하고, 기존 도구와 조합해야 답할 수 있는 질문을 던져보세요.

아래 `my_courses`, `my_budget_won`은 예시로 채워둔 더미 값입니다 -- **원하면 여러분의 실제(또는 가상) 시간표/예산으로 바꿔서 사용해도 됩니다.**

In [ ]:
# 나의 학교생활 정보 (자유롭게 값을 바꿔도 됩니다) + 날짜 계산 헬퍼 (실행만 하면 됩니다)
from datetime import date

my_courses = [
    {"name": "인공지능개론", "credit": 3, "deadline": "2026-09-10", "assignment": "기말 프로젝트"},
    {"name": "선형대수",     "credit": 3, "deadline": "2026-09-05", "assignment": "중간고사"},
    {"name": "영어회화",     "credit": 2, "deadline": "2026-09-20", "assignment": "스피킹 테스트"},
]
my_budget_won = 300000  # 이번 달 남은 용돈/예산 (원)

def days_until(date_str):
    target = date.fromisoformat(date_str)
    return (target - date.today()).days

# 예시: days_until(my_courses[0]["deadline"])

### 여러분의 과제

1. `my_courses`, `my_budget_won`, `days_until()`을 활용하는 **새 도구 함수**를 하나 이상 설계하세요. 예를 들면:
   - `days_until_deadline(course_name)`: 특정 과목의 과제/시험 마감까지 며칠 남았는지 반환
   - `remaining_budget()`: 남은 예산을 반환
   - `list_upcoming_deadlines(within_days)`: N일 이내의 마감이 있는 과목 목록을 반환
   - (자유롭게 다른 아이디어도 좋습니다)
2. 위 본문의 `tools` 리스트 형식(`name`, `description`, `input_schema`)을 참고해서 새 도구를 설명하는 딕셔너리를 만들고, **기존 `tools` 리스트에 추가**하세요 (`tools.append(...)`).
3. 새 도구 함수를 **기존 `TOOL_FUNCTIONS` 딕셔너리에 등록**하세요 (`TOOL_FUNCTIONS["도구이름"] = 함수`).
4. `run_agent(...)`를 호출해서, **새 도구와 기존 도구(계산기/단위변환)를 함께 사용해야 답할 수 있는** 자연어 질문을 던져보세요. 예: *"이번 달 남은 예산을 선형대수 마감일까지 남은 날짜로 나누면 하루에 얼마씩 쓸 수 있어?"*

In [ ]:
# TODO 1: 새 도구 함수를 정의하세요 (my_courses / my_budget_won / days_until 활용).


# TODO 2: 새 도구를 설명하는 딕셔너리를 만들어 tools 리스트에 추가하세요.


# TODO 3: 새 도구 함수를 TOOL_FUNCTIONS에 등록하세요.


# TODO 4: 새 도구 + 기존 도구를 함께 써야 답할 수 있는 질문으로 run_agent()를 호출해보세요.